# Automated Fact-Checking — WhatsApp Chatbot v2
**Wikipedia + DuckDuckGo + NLI + RAG + Citations + Search History**

> Diya Agarwal (2022B4A71705P) | Aditya Gupta (2022B3A70509P) | Aarya Jindal (2022B2A81100P)

### How to use this notebook
1. Run **Cell 1** (install) → restart runtime → run from **Cell 2** onwards
2. After all setup cells run, execute **Cell: Start WhatsApp Server** at the bottom
3. Copy the printed `ngrok` URL, paste into Twilio Sandbox webhook, send a WhatsApp message


## 1. Install Dependencies
> **After this cell runs, go to Runtime → Restart Runtime, then run from Cell 2.**

In [32]:
# ============================================================
# Cell 1 — Install all dependencies
# Run this cell FIRST, then restart runtime, then run from Cell 2.
# ============================================================
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "numpy==1.26.4", "--force-reinstall"], capture_output=True)

!pip -q install -U \
    "numpy==1.26.4" "scipy==1.13.1" "scikit-learn==1.5.2" \
    "transformers==4.47.1" accelerate sentencepiece sacremoses \
    emoji regex pandas pyarrow datasets sentence-transformers \
    ddgs trafilatura beautifulsoup4 requests \
    fastapi uvicorn nest_asyncio \
    lxml_html_clean twilio pyngrok flask \
    wikipedia-api wikipedia rake-nltk \
    keybert nltk

print("=" * 55)
print("All packages installed.")
print("NOW: Runtime > Restart Runtime, then run from Cell 2.")
print("=" * 55)


All packages installed.
NOW: Runtime > Restart Runtime, then run from Cell 2.


## 2. Drive Mount & Project Folders

In [1]:
!pip install indic-transliteration
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

In [2]:
# ============================================================
# Cell 2 — Runtime bootstrap, Drive mount, project folders
# ============================================================
import os, json, math, time, random, hashlib, logging, textwrap
import datetime, warnings, threading
warnings.filterwarnings('ignore', category=DeprecationWarning)
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None

if IN_COLAB:
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount notice:", e)

DEFAULT_ROOT = "/content/drive/MyDrive/whatsapp_factcheck_project" if IN_COLAB else "./whatsapp_factcheck_project"
ROOT_DIR = Path(os.environ.get("WHATSAPP_FACTCHECK_ROOT", DEFAULT_ROOT))

PROJECT_DIRS = {
    "root":        ROOT_DIR,
    "config":      ROOT_DIR / "config",
    "data":        ROOT_DIR / "data",
    "data_aliases":ROOT_DIR / "data" / "aliases",
    "data_corpus": ROOT_DIR / "data" / "corpus",
    "models":      ROOT_DIR / "models",
    "indices":     ROOT_DIR / "indices",
    "outputs":     ROOT_DIR / "outputs",
    "logs":        ROOT_DIR / "logs",
    "cache":       ROOT_DIR / "cache",
    "web_cache":   ROOT_DIR / "cache" / "web"
}
for p in PROJECT_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

LOG_FORMAT = "%(asctime)s [%(levelname)s] %(name)s: %(message)s"
logging.basicConfig(level=logging.INFO, format=LOG_FORMAT)
logger = logging.getLogger("FactCheckBot")
fh = logging.FileHandler(PROJECT_DIRS["logs"] / "factcheck.log")
fh.setFormatter(logging.Formatter(LOG_FORMAT))
logger.addHandler(fh)

print("Project root:", ROOT_DIR)
for k, v in PROJECT_DIRS.items():
    print(f"  {k:14s} -> {v}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/whatsapp_factcheck_project
  root           -> /content/drive/MyDrive/whatsapp_factcheck_project
  config         -> /content/drive/MyDrive/whatsapp_factcheck_project/config
  data           -> /content/drive/MyDrive/whatsapp_factcheck_project/data
  data_aliases   -> /content/drive/MyDrive/whatsapp_factcheck_project/data/aliases
  data_corpus    -> /content/drive/MyDrive/whatsapp_factcheck_project/data/corpus
  models         -> /content/drive/MyDrive/whatsapp_factcheck_project/models
  indices        -> /content/drive/MyDrive/whatsapp_factcheck_project/indices
  outputs        -> /content/drive/MyDrive/whatsapp_factcheck_project/outputs
  logs           -> /content/drive/MyDrive/whatsapp_factcheck_project/logs
  cache          -> /content/drive/MyDrive/whatsapp_factcheck_project/cache
  web_cache      -> /content/drive/M

## 3. Config Helpers & Secret Loader

In [3]:
# ============================================================
# Cell 3 — Config helpers + secret loader
# ============================================================

def write_json_if_missing(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with open(path, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
        print(f"Created: {path}")
    else:
        print(f"Already exists: {path}")

def load_json(path):
    with open(Path(path), "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def try_get_secret(name: str, default=None):
    val = os.environ.get(name)
    if val:
        return val
    try:
        from google.colab import userdata
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    except Exception:
        pass
    return default

print("Config helpers loaded.")


Config helpers loaded.


## 4. Build Config Files & Load Variables

In [4]:
# ============================================================
# Cell 4 — Create/merge config files + expose runtime variables
# ============================================================
_DEF_MODELS = {
    "gemini_model": "gemini-2.5-flash",
    "nllb_model": "facebook/nllb-200-distilled-1.3B",
    "nli_model": "roberta-large-mnli",
    "emotion_model": "SamLowe/roberta-base-go_emotions",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2"
}
_DEF_THRESH = {
    "romanized_prob_threshold": 0.35,
    "emotion_pick_threshold": 0.30,
    "meaning_nli_threshold": 0.80,
    "retrieval_top_k": 5,
    "web_search_top_k": 6,
    "web_min_similarity": 0.20,
    "corpus_min_similarity": 0.20,
    "high_conf_corpus_similarity": 0.72,
    "max_new_tokens": 256,
    "max_article_chars": 12000,
    "max_snippet_chars": 600,
    "max_snippets_per_doc": 3,
    "query_expansion_top_k": 3,
    "top_similar_k": 5
}
_DEF_PIPE = {
    "default_devanagari_lang": "hin_Deva",
    "use_gemini_if_available": True,
    "seed_demo_corpus_if_empty": True,
    "run_optional_emotion_finetune": False,
    "enable_query_expansion": True,
    "enable_indian_aliasing": True
}
_DEF_PATHS = {
    "emotion_model_dir": "/content/drive/MyDrive/goemotions_roberta" if IN_COLAB else str(PROJECT_DIRS["models"] / "goemotions_roberta"),
    "corpus_csv_path": str(PROJECT_DIRS["data_corpus"] / "factcheck_corpus.csv"),
    "corpus_parquet_path": str(PROJECT_DIRS["data_corpus"] / "factcheck_corpus.parquet"),
    "embeddings_npy_path": str(PROJECT_DIRS["indices"] / "factcheck_embeddings.npy"),
    "metadata_json_path": str(PROJECT_DIRS["indices"] / "factcheck_metadata.json"),
    "web_cache_dir": str(PROJECT_DIRS["web_cache"])
}
_DEF_SOURCES = {"trusted_domains": [
    "who.int", "mohfw.gov.in", "pib.gov.in", "cdc.gov", "nih.gov",
    "reuters.com", "apnews.com", "bbc.com", "indianexpress.com", "thehindu.com",
    "factcheck.org", "snopes.com", "altnews.in", "boomlive.in"
]}
_DEF_VERDICTS = {
    "allowed_verdicts": ["supported","refuted","misleading","mixed","unverified","not_enough_evidence"],
    "required_nonempty_fields": ["claim_text_original","verdict_label","source_name","source_url","article_title"]
}
_DEF_CLAIM_CFG = {
    "min_claim_chars": 12, "min_tokens": 3,
    "max_candidates_per_message": 10, "max_checkworthy_claims": 5, "base_threshold": 0.45,
    "drop_meta_phrases": ["forwarded many times","forwarded","breaking","share urgently","share this",
        "please share","must share","viral message","urgent","read till end","watch till end",
        "click here","join now","subscribe now"],
    "factual_cue_verbs": ["is","are","was","were","causes","cause","cures","cure","prevents","prevent",
        "kills","kill","died","dies","dead","announced","banned","approved","contains","linked",
        "spreads","works","fails","increases","decreases","results in","leads to","due to"],
    "non_checkworthy_starters": ["i think","i feel","in my opinion","maybe","perhaps","please",
        "share","wow","omg","breaking","urgent"]
}
_DEF_ALIASES = {
    "Pappu":"Rahul Gandhi","Feku":"Narendra Modi","NaMo":"Narendra Modi",
    "RaGa":"Rahul Gandhi","MMS":"Manmohan Singh","AK49":"Arvind Kejriwal",
    "Didi":"Mamata Banerjee","Behenji":"Mayawati","Yogi ji":"Yogi Adityanath",
    "Modiji":"Narendra Modi","Kejru":"Arvind Kejriwal"
}

def _merge_cfg(path, defaults):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    on_disk = {}
    if path.exists():
        try:
            with open(path) as f: on_disk = json.load(f)
        except: pass
    merged = {**defaults, **on_disk}
    with open(path, "w") as f: json.dump(merged, f, indent=2)
    return merged

def _safe_load(path, fallback=None):
    try:
        with open(Path(path)) as f: return json.load(f)
    except: return fallback or {}

MODELS      = _merge_cfg(PROJECT_DIRS["config"]/"models.json",     _DEF_MODELS)
THRESHOLDS  = _merge_cfg(PROJECT_DIRS["config"]/"thresholds.json", _DEF_THRESH)
PIPELINE    = _merge_cfg(PROJECT_DIRS["config"]/"pipeline.json",   _DEF_PIPE)
PATHS       = _merge_cfg(PROJECT_DIRS["config"]/"paths.json",      _DEF_PATHS)
SOURCES     = _merge_cfg(PROJECT_DIRS["config"]/"sources.json",    _DEF_SOURCES)
VERDICT_CFG = _merge_cfg(PROJECT_DIRS["config"]/"verdicts.json",   _DEF_VERDICTS)
CLAIM_CFG   = _merge_cfg(PROJECT_DIRS["config"]/"claim_extraction.json", _DEF_CLAIM_CFG)
INDIAN_ALIASES    = _merge_cfg(PROJECT_DIRS["config"]/"aliases.json",         _DEF_ALIASES)
EMOJI_EMOTION_MAP = _safe_load(PROJECT_DIRS["config"]/"emoji_emotion_map.json", {})

# Expose runtime constants
GEMINI_MODEL_NAME       = MODELS["gemini_model"]
NLLB_MODEL_NAME         = MODELS["nllb_model"]
NLI_MODEL_NAME          = MODELS["nli_model"]
EMOTION_MODEL_NAME      = MODELS["emotion_model"]
EMBEDDING_MODEL_NAME    = MODELS["embedding_model"]
ROMANIZED_PROB_THRESHOLD    = THRESHOLDS["romanized_prob_threshold"]
EMOTION_PICK_THRESHOLD      = THRESHOLDS["emotion_pick_threshold"]
MEANING_NLI_THRESHOLD       = THRESHOLDS["meaning_nli_threshold"]
RETRIEVAL_TOP_K             = THRESHOLDS["retrieval_top_k"]
WEB_SEARCH_TOP_K            = THRESHOLDS["web_search_top_k"]
WEB_MIN_SIMILARITY          = THRESHOLDS["web_min_similarity"]
CORPUS_MIN_SIMILARITY       = THRESHOLDS["corpus_min_similarity"]
HIGH_CONF_CORPUS_SIMILARITY = THRESHOLDS["high_conf_corpus_similarity"]
MAX_NEW_TOKENS      = THRESHOLDS["max_new_tokens"]
MAX_ARTICLE_CHARS   = THRESHOLDS["max_article_chars"]
MAX_SNIPPET_CHARS   = THRESHOLDS["max_snippet_chars"]
MAX_SNIPPETS_PER_DOC= THRESHOLDS["max_snippets_per_doc"]
TOP_SIMILAR_K       = THRESHOLDS.get("top_similar_k", 5)
DEFAULT_DEVANAGARI_LANG = PIPELINE["default_devanagari_lang"]
USE_GEMINI_IF_AVAILABLE = PIPELINE["use_gemini_if_available"]
SEED_DEMO_CORPUS_IF_EMPTY = PIPELINE["seed_demo_corpus_if_empty"]
ENABLE_QUERY_EXPANSION  = PIPELINE.get("enable_query_expansion", True)
ENABLE_INDIAN_ALIASING  = PIPELINE.get("enable_indian_aliasing", True)
EMO_MODEL_DIR       = PATHS["emotion_model_dir"]
CORPUS_CSV_PATH     = PATHS["corpus_csv_path"]
CORPUS_PARQUET_PATH = PATHS["corpus_parquet_path"]
EMBEDDINGS_NPY_PATH = PATHS["embeddings_npy_path"]
METADATA_JSON_PATH  = PATHS["metadata_json_path"]
WEB_CACHE_DIR       = PATHS["web_cache_dir"]

print("All configs loaded.")
print(f"  Embedding : {EMBEDDING_MODEL_NAME}")
print(f"  NLI       : {NLI_MODEL_NAME}")
print(f"  Corpus CSV: {CORPUS_CSV_PATH}")


All configs loaded.
  Embedding : sentence-transformers/all-MiniLM-L6-v2
  NLI       : roberta-large-mnli
  Corpus CSV: /content/drive/MyDrive/whatsapp_factcheck_project/data/corpus/factcheck_corpus.csv


## 5. API Secrets (Gemini + Twilio)
> Store `GEMINI_API_KEY`, `TWILIO_ACCOUNT_SID`, `TWILIO_AUTH_TOKEN`, `TWILIO_WHATSAPP_NUMBER` in Colab Secrets (key icon on left sidebar).

In [5]:
# ============================================================
# Cell 5 — Load API secrets + VALIDATE Twilio immediately
# ============================================================
GEMINI_API_KEY         = try_get_secret("GEMINI_API_KEY")
TWILIO_ACCOUNT_SID     = try_get_secret("TWILIO_ACCOUNT_SID")
TWILIO_AUTH_TOKEN      = try_get_secret("TWILIO_AUTH_TOKEN")
TWILIO_WHATSAPP_NUMBER = try_get_secret("TWILIO_WHATSAPP_NUMBER")

GEMINI_AVAILABLE  = bool(GEMINI_API_KEY) and USE_GEMINI_IF_AVAILABLE
TWILIO_CONFIGURED = bool(TWILIO_ACCOUNT_SID and TWILIO_AUTH_TOKEN and TWILIO_WHATSAPP_NUMBER)

if GEMINI_AVAILABLE:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        print("Gemini: available")
    except Exception as e:
        GEMINI_AVAILABLE = False
        print("Gemini: disabled:", e)
else:
    print("Gemini: not configured")

print()
print("=" * 55)
print("TWILIO CONFIGURATION CHECK")
print("=" * 55)
print(f"  ACCOUNT_SID    : {'SET (' + TWILIO_ACCOUNT_SID[:8] + '...)' if TWILIO_ACCOUNT_SID else 'MISSING'}")
print(f"  AUTH_TOKEN     : {'SET' if TWILIO_AUTH_TOKEN else 'MISSING'}")
print(f"  WHATSAPP_NUMBER: {TWILIO_WHATSAPP_NUMBER or 'MISSING'}")
print()

if TWILIO_CONFIGURED:
    if not TWILIO_WHATSAPP_NUMBER.startswith("whatsapp:"):
        TWILIO_WHATSAPP_NUMBER = "whatsapp:" + TWILIO_WHATSAPP_NUMBER
        print(f"  Auto-fixed number to: {TWILIO_WHATSAPP_NUMBER}")
    try:
        from twilio.rest import Client as _TC
        _client = _TC(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
        _acct   = _client.api.accounts(TWILIO_ACCOUNT_SID).fetch()
        print(f"  Twilio REST API : CONNECTED ({_acct.friendly_name})")
        print("  Status          : READY TO SEND MESSAGES")
    except Exception as e:
        print(f"  Twilio REST API : FAILED - {e}")
        print("  Fix your ACCOUNT_SID / AUTH_TOKEN in Colab Secrets")
else:
    print("  TWILIO NOT CONFIGURED - replies will NOT work!")
    print("  Add to Colab Secrets (key icon on left sidebar):")
    print("    TWILIO_ACCOUNT_SID     = ACxxxxxxxx")
    print("    TWILIO_AUTH_TOKEN      = your_token")
    print("    TWILIO_WHATSAPP_NUMBER = whatsapp:+14155238886")
print("=" * 55)


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini: available

TWILIO CONFIGURATION CHECK
  ACCOUNT_SID    : SET (ACcb4c5e...)
  AUTH_TOKEN     : SET
  WHATSAPP_NUMBER: +14155238886

  Auto-fixed number to: whatsapp:+14155238886
  Twilio REST API : CONNECTED (My first Twilio account)
  Status          : READY TO SEND MESSAGES


## 6. Core Imports & Lazy Model Loaders

In [6]:
# ============================================================
# Cell 6 — Core imports + lazy model caches
# ============================================================
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
import numpy as np
import pandas as pd
import torch
import requests
import regex as re
from pathlib import Path
from bs4 import BeautifulSoup

try:
    from ddgs import DDGS
except ImportError:
    try:
        from duckduckgo_search import DDGS
    except ImportError:
        DDGS = None
        print("WARNING: ddgs not installed.")

import trafilatura
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification, pipeline
)

device = 0 if torch.cuda.is_available() else -1
torch_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", torch_device)

_NLLB_TOKENIZER = None; _NLLB_MODEL = None
_NLI_TOKENIZER  = None; _NLI_MODEL  = None
_EMOTION_TOKENIZER = None; _EMOTION_MODEL = None
_EMOTION_LABELS = None;  _EMOTION_THRESHOLD = None
_EMBEDDER = None

from transformers import NllbTokenizer, AutoModelForSeq2SeqLM

def get_nllb():
    global _NLLB_TOKENIZER, _NLLB_MODEL
    if _NLLB_TOKENIZER is None:
        print("Loading NLLB:", NLLB_MODEL_NAME)
        _NLLB_TOKENIZER = NllbTokenizer.from_pretrained(NLLB_MODEL_NAME)
        _NLLB_MODEL = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_NAME)
        if torch.cuda.is_available():
            _NLLB_MODEL = _NLLB_MODEL.to(torch_device)
        _NLLB_MODEL.eval()
    return _NLLB_TOKENIZER, _NLLB_MODEL

def get_nli():
    global _NLI_TOKENIZER, _NLI_MODEL
    if _NLI_TOKENIZER is None:
        print("Loading NLI model:", NLI_MODEL_NAME)
        _NLI_TOKENIZER = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
        _NLI_MODEL = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
        if torch.cuda.is_available():
            _NLI_MODEL = _NLI_MODEL.to(torch_device)
        _NLI_MODEL.eval()
    return _NLI_TOKENIZER, _NLI_MODEL

def get_emotion_model():
    global _EMOTION_TOKENIZER, _EMOTION_MODEL, _EMOTION_THRESHOLD, _EMOTION_LABELS
    if _EMOTION_TOKENIZER is not None:
        return _EMOTION_TOKENIZER, _EMOTION_MODEL, _EMOTION_LABELS, _EMOTION_THRESHOLD
    local_path  = Path(EMO_MODEL_DIR)
    has_weights = any((local_path / f).exists() for f in ["model.safetensors","pytorch_model.bin"])
    model_ref   = str(local_path) if (local_path.exists() and has_weights) else EMOTION_MODEL_NAME
    print("Loading GoEmotions from:", model_ref)
    _EMOTION_TOKENIZER = AutoTokenizer.from_pretrained(model_ref)
    _EMOTION_MODEL     = AutoModelForSequenceClassification.from_pretrained(model_ref)
    if torch.cuda.is_available():
        _EMOTION_MODEL = _EMOTION_MODEL.to(torch_device)
    _EMOTION_MODEL.eval()
    thr_path = local_path / "threshold.txt"
    _EMOTION_THRESHOLD = float(thr_path.read_text().strip()) if thr_path.exists() else EMOTION_PICK_THRESHOLD
    try:
        from datasets import load_dataset
        _EMOTION_LABELS = load_dataset("go_emotions")["train"].features["labels"].feature.names
    except Exception:
        _EMOTION_LABELS = None
    return _EMOTION_TOKENIZER, _EMOTION_MODEL, _EMOTION_LABELS, _EMOTION_THRESHOLD

def get_embedder():
    global _EMBEDDER
    if _EMBEDDER is None:
        from sentence_transformers import SentenceTransformer
        _EMBEDDER = SentenceTransformer(EMBEDDING_MODEL_NAME)
    return _EMBEDDER

print("Model loaders ready.")


Torch device: cuda
Model loaders ready.


## 7. NLI Probability Helper

In [7]:
# ============================================================
# Cell 7 — NLI inference helper
# ============================================================
def nli_probs(premise: str, hypothesis: str) -> dict:
    tok, mdl = get_nli()
    enc = tok(premise[:512], hypothesis[:512], return_tensors="pt",
               truncation=True, max_length=512)
    enc = {k: v.to(torch_device) for k, v in enc.items()}
    with torch.no_grad():
        logits = mdl(**enc).logits
    probs  = torch.softmax(logits, dim=-1)[0].tolist()
    labels = [mdl.config.id2label[i] for i in range(len(probs))]
    return {l: p for l, p in zip(labels, probs)}

print("NLI helper ready.")


NLI helper ready.


## 8. Text Preprocessing Pipeline

In [8]:
def normalize_roman_hindi(text):
    text = " " + text.lower().strip() + " "
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # small normalization only for spelling variants, not translation
    norm_map = {
        "nhi": "nahi",
        "ni": "nahi",
        "mei": "mein",
        "mai": "main",
        "vo": "woh",
        "ye": "yeh",
        "kr": "kar",
        "kiyaa": "kiya",
        "huaa": "hua",
        "h": "hai"
    }

    tokens = text.split()
    tokens = [norm_map.get(tok, tok) for tok in tokens]
    return " ".join(tokens)

def roman_hindi_to_devanagari(text):
    try:
        norm_text = normalize_roman_hindi(text)
        dev_text = transliterate(norm_text, sanscript.ITRANS, sanscript.DEVANAGARI)
        return dev_text.strip() if dev_text else text
    except Exception as e:
        logger.warning("Roman->Devanagari transliteration failed: %s", e)
        return text

def translation_looks_bad(src_text, translated_text):
    src = re.sub(r"\s+", " ", src_text.lower()).strip()
    out = re.sub(r"\s+", " ", translated_text.lower()).strip()

    if not out:
        return True
    if len(out) < 5:
        return True
    if out == src:
        return True

    # if too much devanagari remains in "English" output, translation likely failed
    if HINDI_CHARS.search(out):
        return True

    return False

In [9]:
# ============================================================
# Cell 8 — Multilingual Preprocessing (FIXED)
#
# FIXES:
#   - Hindi (Devanagari) text properly translated via NLLB
#   - Hinglish (Roman-script Hindi) detected and translated
#   - Bengali script supported
#   - Translation shown in WhatsApp reply so user knows it worked
#   - Aliases applied AFTER translation for accuracy
#   - Faster: translation only triggered when actually needed
# ============================================================
import unicodedata

HINDI_CHARS   = re.compile(r'[\u0900-\u097F]')
BENGALI_CHARS = re.compile(r'[\u0980-\u09FF]')
LATIN_SCRIPT  = re.compile(r'[a-zA-Z]')

# Common Hinglish/romanized Hindi words that signal Hindi content
HINGLISH_SIGNALS = {
    "mai", "mein", "ne", "ke", "ka", "ki", "ko", "se", "par", "aur",
    "hai", "hain", "tha", "the", "thi", "kya", "nahi", "nhi", "bhi",
    "koi", "sab", "woh", "yeh", "ye", "vo", "jo", "jab", "tab",
    "maare", "maara", "kiya", "kiye", "karke", "gaye", "aye", "aaye",
    "goals", "trophies", "jeet", "haar", "score", "khela", "khele"
}


def detect_script(text):
    if HINDI_CHARS.search(text):
        return "devanagari"
    if BENGALI_CHARS.search(text):
        return "bengali"

    if LATIN_SCRIPT.search(text):
        tokens = re.findall(r"[a-zA-Z]+", text.lower())
        if not tokens:
            return "unknown"

        signal_hits = sum(tok in HINGLISH_SIGNALS for tok in tokens)
        signal_ratio = signal_hits / max(len(tokens), 1)

        # Hinglish if enough Hindi-style signals appear
        if signal_hits >= 2 or signal_ratio >= 0.20:
            return "hinglish"

        return "latin"

    return "unknown"

def apply_indian_aliases(text):
    if not ENABLE_INDIAN_ALIASING:
        return text
    for alias, real in INDIAN_ALIASES.items():
        text = re.sub(r'\b' + re.escape(alias) + r'\b', real, text, flags=re.IGNORECASE)
    return text

def clean_whatsapp_text(text):
    text = re.sub(r'(Forwarded|\u092b\u093c\u093c\u0949\u0930\u0935\u0930\u094d\u0921 \u0915\u093f\u092f\u093e \u0917\u092f\u093e).*?\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'([!?.]){3,}', r'\1\1', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def translate_to_english(text, src_lang="hin_Deva"):
    try:
        tok, mdl = get_nllb()
        if tok is None or mdl is None:
            return text

        tok.src_lang = src_lang
        inputs = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )
        inputs = {k: v.to(torch_device) for k, v in inputs.items()}

        tgt_lang_id = tok.convert_tokens_to_ids("eng_Latn")

        outputs = mdl.generate(
            **inputs,
            forced_bos_token_id=tgt_lang_id,
            max_new_tokens=256,
            num_beams=2,
            early_stopping=True
        )

        translated = tok.decode(outputs[0], skip_special_tokens=True).strip()
        return translated if translated else text

    except Exception as e:
        logger.warning("Translation failed: %s", e)
        return text

def preprocess_message(raw_text):
    cleaned = clean_whatsapp_text(raw_text)
    cleaned = apply_indian_aliases(cleaned)
    script = detect_script(cleaned)

    translated_en = ""
    transliterated_text = ""

    if script == "devanagari":
        translated_en = translate_to_english(cleaned, src_lang=DEFAULT_DEVANAGARI_LANG)

    elif script == "bengali":
        translated_en = translate_to_english(cleaned, src_lang="ben_Beng")

    elif script == "hinglish":
        transliterated_text = roman_hindi_to_devanagari(cleaned)
        translated_en = translate_to_english(transliterated_text, src_lang="hin_Deva")

        # if transliteration+translation fails, keep original text
        if translation_looks_bad(cleaned, translated_en):
            translated_en = cleaned

    canonical_en = translated_en or cleaned
    canonical_en = apply_indian_aliases(canonical_en)

    return {
        "raw": raw_text,
        "cleaned_raw": cleaned,
        "script": script,
        "transliterated_text": transliterated_text,
        "translated_en": translated_en,
        "canonical_en": canonical_en,
        "was_translated": bool(translated_en and translated_en != cleaned)
    }

def preprocess_message_debug(raw_text):
    return {"dict_output": preprocess_message(raw_text)}

print("Multilingual preprocessing ready (Hindi/Hinglish/Bengali/English).")


Multilingual preprocessing ready (Hindi/Hinglish/Bengali/English).


## 9. Claim Extraction

In [10]:
# ============================================================
# Cell 9 — Claim extraction
# ============================================================
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

from rake_nltk import Rake
from keybert import KeyBERT

_KEYBERT = None
def get_keybert():
    global _KEYBERT
    if _KEYBERT is None:
        _KEYBERT = KeyBERT(model=EMBEDDING_MODEL_NAME)
    return _KEYBERT

def score_claim_checkworthy(claim_text):
    score = 0.0
    text_l = claim_text.lower()
    for verb in CLAIM_CFG.get("factual_cue_verbs", []):
        if verb in text_l:
            score += 0.15
    for starter in CLAIM_CFG.get("non_checkworthy_starters", []):
        if text_l.startswith(starter):
            score -= 0.5
    for phrase in CLAIM_CFG.get("drop_meta_phrases", []):
        if phrase in text_l:
            score -= 0.8
    if re.search(r'\b\d+(\.\d+)?%?\b', claim_text):
        score += 0.10
    if len(claim_text.split()) >= 4:
        score += 0.10
    return round(min(max(score, 0.0), 1.0), 4)

def extract_claim_candidates(raw_text):
    prep  = preprocess_message(raw_text)
    wtext = prep.get("canonical_en","").strip() or prep.get("cleaned_raw","").strip() or raw_text
    try:
        sentences = nltk.sent_tokenize(wtext)
    except Exception:
        sentences = [s.strip() for s in re.split(r'[.!?]', wtext) if s.strip()]
    candidates = []
    min_chars  = CLAIM_CFG.get("min_claim_chars", 12)
    min_tokens = CLAIM_CFG.get("min_tokens", 3)
    for sent in sentences:
        sent = sent.strip()
        if len(sent) < min_chars or len(sent.split()) < min_tokens:
            continue
        score = score_claim_checkworthy(sent)
        candidates.append({"claim_text": sent, "checkworthy_score": score})
    candidates.sort(key=lambda x: x["checkworthy_score"], reverse=True)
    base_thr = CLAIM_CFG.get("base_threshold", 0.45)
    max_cw   = CLAIM_CFG.get("max_checkworthy_claims", 5)
    checkworthy = [c for c in candidates if c["checkworthy_score"] >= base_thr][:max_cw]
    if not checkworthy and candidates:
        checkworthy = candidates[:1]
    return {
        "working_text_for_claims": wtext,
        "all_candidates":          candidates,
        "checkworthy_claims":      checkworthy
    }

print("Claim extraction ready.")


Claim extraction ready.


## 10. Fact-Check Corpus & Retrieval

In [11]:
# ============================================================
# Cell 10 — Corpus management + semantic retrieval
# ============================================================
CORPUS_SCHEMA = {
    "fact_id":               "",
    "claim_text_original":   "",
    "claim_text_normalized": "",
    "verdict_label":         "",
    "source_name":           "",
    "source_url":            "",
    "article_title":         "",
    "published_date":        "",
    "article_text":          "",
    "evidence_snippets":     "",
    "language":              "en",
    "tags":                  ""
}

def ensure_corpus_schema(df):
    for col, default in CORPUS_SCHEMA.items():
        if col not in df.columns:
            df[col] = default
    if "fact_id" in df.columns and df["fact_id"].eq("").all():
        df["fact_id"] = [f"fc_{i:05d}" for i in range(len(df))]
    return df[list(CORPUS_SCHEMA.keys())]

def seed_demo_corpus():
    return pd.DataFrame([
        {"fact_id":"demo_001","claim_text_original":"Drinking cow urine cures COVID-19",
         "claim_text_normalized":"","verdict_label":"refuted","source_name":"WHO",
         "source_url":"https://www.who.int/","article_title":"COVID-19 Mythbusters",
         "published_date":"2021-01-01","article_text":"","evidence_snippets":
         "WHO: No evidence that animal urine treats coronavirus.","language":"en","tags":"demo"},
        {"fact_id":"demo_002","claim_text_original":"India launched Chandrayaan-3 to the Moon in 2023",
         "claim_text_normalized":"","verdict_label":"supported","source_name":"ISRO",
         "source_url":"https://www.isro.gov.in/","article_title":"Chandrayaan-3 Mission",
         "published_date":"2023-08-23","article_text":"","evidence_snippets":
         "ISRO successfully landed Chandrayaan-3 on the Moon on 23 August 2023.","language":"en","tags":"demo"},
        {"fact_id":"demo_003","claim_text_original":"5G towers spread coronavirus",
         "claim_text_normalized":"","verdict_label":"refuted","source_name":"Reuters",
         "source_url":"https://reuters.com/","article_title":"Fact Check: 5G and COVID-19",
         "published_date":"2020-04-01","article_text":"","evidence_snippets":
         "Viruses cannot travel on radio waves. 5G has no link to COVID-19.","language":"en","tags":"demo"}
    ])

def load_factcheck_corpus():
    csv_path = Path(CORPUS_CSV_PATH)
    if csv_path.exists():
        try:
            df = pd.read_csv(csv_path).fillna("")
            return ensure_corpus_schema(df)
        except Exception:
            pass
    if SEED_DEMO_CORPUS_IF_EMPTY:
        df = ensure_corpus_schema(seed_demo_corpus())
        save_factcheck_corpus(df)
        return df
    return ensure_corpus_schema(pd.DataFrame())

def save_factcheck_corpus(df):
    df = ensure_corpus_schema(df)
    df.to_csv(CORPUS_CSV_PATH, index=False)

def enrich_corpus_with_normalized_claims(df):
    if "claim_text_normalized" not in df.columns:
        df["claim_text_normalized"] = ""
    mask = df["claim_text_normalized"].eq("")
    for idx in df[mask].index:
        raw = str(df.at[idx, "claim_text_original"])
        prep = preprocess_message(raw)
        df.at[idx, "claim_text_normalized"] = (
            prep.get("canonical_en","").strip() or
            prep.get("cleaned_raw","").strip() or raw
        )
    return df

def build_factcheck_retrieval_artifacts():
    df = load_factcheck_corpus()
    df = enrich_corpus_with_normalized_claims(df)
    if len(df) == 0:
        print("Corpus is empty.")
        return None
    texts = df["claim_text_normalized"].fillna("").astype(str).tolist()
    embeddings = get_embedder().encode(texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    embeddings = np.asarray(embeddings, dtype=np.float32)
    save_factcheck_corpus(df)
    np.save(EMBEDDINGS_NPY_PATH, embeddings)
    save_json(METADATA_JSON_PATH, {"embedding_model_name": EMBEDDING_MODEL_NAME,
        "num_records": int(len(df)), "corpus_csv_path": CORPUS_CSV_PATH})
    return {"df": df, "embeddings": embeddings}

def load_factcheck_retrieval_artifacts():
    csv_path = Path(CORPUS_CSV_PATH)
    emb_path = Path(EMBEDDINGS_NPY_PATH)
    if not csv_path.exists() or not emb_path.exists():
        return None
    df = ensure_corpus_schema(pd.read_csv(csv_path).fillna(""))
    embeddings = np.load(emb_path)
    return {"df": df, "embeddings": embeddings}

def preprocess_query_claim(claim_text):
    prep = preprocess_message(claim_text)
    return prep.get("canonical_en","").strip() or prep.get("cleaned_raw","").strip() or claim_text.strip()

def cosine_top_k(query_vec, matrix, top_k):
    sims = matrix @ query_vec
    idx  = np.argsort(-sims)[:min(top_k, len(sims))]
    return idx, sims[idx]

def retrieve_similar_factchecks_for_claim(claim_text, top_k=None, min_similarity=None):
    top_k          = top_k or RETRIEVAL_TOP_K
    min_similarity = CORPUS_MIN_SIMILARITY if min_similarity is None else min_similarity
    arts = load_factcheck_retrieval_artifacts()
    if arts is None:
        return {"query_claim_original": claim_text, "matches": []}
    df, embeddings = arts["df"], arts["embeddings"]
    query_norm = preprocess_query_claim(claim_text)
    query_vec  = get_embedder().encode([query_norm], normalize_embeddings=True)
    query_vec  = np.asarray(query_vec[0], dtype=np.float32)
    idxs, sims = cosine_top_k(query_vec, embeddings, top_k=top_k)
    matches = []
    for idx, sim in zip(idxs, sims):
        score = float(sim)
        if score < min_similarity:
            continue
        row = df.iloc[int(idx)].to_dict()
        matches.append({
            "similarity": round(score, 4),
            **{k: row.get(k,"") for k in ["fact_id","claim_text_original","claim_text_normalized",
               "verdict_label","source_name","source_url","article_title","published_date",
               "evidence_snippets","tags"]}
        })
    return {"query_claim_original": claim_text, "query_claim_normalized": query_norm, "matches": matches}

# Build or load
artifacts = load_factcheck_retrieval_artifacts()
if artifacts is None:
    artifacts = build_factcheck_retrieval_artifacts()
print("Artifacts ready:", artifacts is not None)


Artifacts ready: True


In [12]:
# ============================================================
# Cell 10b — CLEAR BAD CORPUS ENTRIES (Run once after loading)
#
# Your corpus has wrong auto-learned entries from previous sessions.
# This cell wipes them and keeps only the 3 clean demo entries.
# The corpus is now used ONLY for displaying similar past claims,
# NOT for making verdict decisions.
# ============================================================
import shutil
from pathlib import Path

def reset_corpus_to_clean_state():
    corpus_path = Path(CORPUS_CSV_PATH)
    embeddings_path = Path(EMBEDDINGS_NPY_PATH)
    web_cache_path  = Path(WEB_CACHE_DIR)

    # Backup old corpus if it exists
    if corpus_path.exists():
        backup = corpus_path.with_suffix(".bak.csv")
        shutil.copy(corpus_path, backup)
        print(f"Backed up old corpus to: {backup}")

    # Wipe auto-learned entries, keep only demo entries
    df = load_factcheck_corpus()
    before = len(df)
    if "tags" in df.columns:
        df = df[~df["tags"].str.contains("auto-learned", na=False)]
    after = len(df)
    print(f"Removed {before - after} auto-learned entries ({after} clean entries remain)")

    # If empty, reseed with clean demo data
    if len(df) == 0:
        df = ensure_corpus_schema(seed_demo_corpus())
        print("Reseeded with 3 clean demo entries")

    save_factcheck_corpus(df)

    # Delete stale embeddings so they get rebuilt from clean corpus
    if embeddings_path.exists():
        embeddings_path.unlink()
        print("Deleted stale embeddings (will rebuild on next query)")

    # Clear web cache so stale article fetches don't affect results
    if web_cache_path.exists():
        cleared = 0
        for f in web_cache_path.glob("*.json"):
            f.unlink()
            cleared += 1
        print(f"Cleared {cleared} stale web cache files")

    # Rebuild clean retrieval artifacts
    artifacts = build_factcheck_retrieval_artifacts()
    print(f"Rebuilt artifacts: {artifacts is not None}")
    print()
    print("Corpus reset complete. Verdict engine now uses LIVE evidence only.")

reset_corpus_to_clean_state()


Backed up old corpus to: /content/drive/MyDrive/whatsapp_factcheck_project/data/corpus/factcheck_corpus.bak.csv
Removed 0 auto-learned entries (3 clean entries remain)
Deleted stale embeddings (will rebuild on next query)
Cleared 1 stale web cache files


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Rebuilt artifacts: True

Corpus reset complete. Verdict engine now uses LIVE evidence only.


## 11. Trusted Source Verification (Wikipedia + Web + Cross-Verification)

In [13]:
# ============================================================
# Cell 11 — Fast Wikipedia + Web Search
#
# SPEED FIXES (main cause of 90s timeouts):
#   - Wikipedia + web search run in PARALLEL threads
#   - Web search uses DuckDuckGo snippets directly (no article fetch)
#   - Wikipedia limited to 4 sentences max
#   - Single focused search query (not 3 queries)
#   - Timeout on each network call (5s Wikipedia, 8s web)
# ============================================================
import concurrent.futures
from urllib.parse import urlparse

def domain_of(url):
    try:
        return urlparse(url).netloc.lower()
    except Exception:
        return ""

def fetch_wikipedia(claim_text, sentences=4):
    try:
        import wikipedia
        wikipedia.set_lang("en")
        results = wikipedia.search(claim_text, results=2)
        if not results:
            return None
        for title in results[:2]:
            try:
                page    = wikipedia.page(title, auto_suggest=False)
                summary = wikipedia.summary(title, sentences=sentences, auto_suggest=False)
                return {"source":"Wikipedia","title":page.title,
                        "url":page.url,"text":summary,"type":"wikipedia"}
            except Exception:
                continue
    except Exception as e:
        logger.warning("Wikipedia fetch failed: %s", e)
    return None

def fetch_web_results(claim_text, max_results=5):
    # ONE focused query only — much faster than 3 queries
    if DDGS is None:
        return []
    results, seen_urls = [], set()
    query = f"fact check {claim_text}"
    try:
        with DDGS() as ddgs:
            hits = list(ddgs.text(query, max_results=max_results))
            for h in hits:
                url = h.get("href") or h.get("url","")
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    results.append({
                        "source": h.get("source") or domain_of(url),
                        "title":  h.get("title",""),
                        "url":    url,
                        "text":   h.get("body",""),   # use snippet directly, no full-page fetch
                        "type":   "web"
                    })
    except Exception as e:
        logger.warning("Web search failed: %s", e)
    return results[:max_results]

def fetch_web_results_broad(claim_text, max_results=4):
    # Fallback: plain search without "fact check" prefix
    if DDGS is None:
        return []
    results, seen_urls = [], set()
    try:
        with DDGS() as ddgs:
            hits = list(ddgs.text(claim_text, max_results=max_results))
            for h in hits:
                url = h.get("href") or h.get("url","")
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    results.append({
                        "source": h.get("source") or domain_of(url),
                        "title":  h.get("title",""),
                        "url":    url,
                        "text":   h.get("body",""),
                        "type":   "web"
                    })
    except Exception as e:
        logger.warning("Broad web search failed: %s", e)
    return results

def fetch_wikipedia_and_web_parallel(claim_text):
    # Run Wikipedia + web search at the same time — saves ~5-10 seconds
    wiki_result = None
    web_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as ex:
        wiki_future = ex.submit(fetch_wikipedia, claim_text, 4)
        web_future  = ex.submit(fetch_web_results, claim_text, 5)
        try:
            wiki_result = wiki_future.result(timeout=12)
        except Exception as e:
            logger.warning("Wikipedia parallel fetch failed: %s", e)
        try:
            web_results = web_future.result(timeout=12)
        except Exception as e:
            logger.warning("Web parallel fetch failed: %s", e)
    # If no fact-check results, try broad fallback
    if not web_results:
        try:
            web_results = fetch_web_results_broad(claim_text, max_results=4)
        except Exception:
            pass
    return wiki_result, web_results

def cross_verify_sources(wiki_result, web_results, claim_text):
    all_texts = []
    if wiki_result and wiki_result.get("text"):
        all_texts.append(wiki_result["text"])
    for r in web_results[:3]:
        if r.get("text"):
            all_texts.append(r["text"][:500])
    if len(all_texts) < 2:
        return 0.0, "Only one source — cannot cross-verify."
    try:
        emb  = get_embedder()
        vecs = emb.encode(all_texts, normalize_embeddings=True)
        sims = [float(vecs[i] @ vecs[j])
                for i in range(len(vecs)) for j in range(i+1, len(vecs))]
        avg_sim = float(sum(sims)/len(sims)) if sims else 0.0
        return avg_sim, f"Cross-verified {len(all_texts)} sources (avg sim: {avg_sim:.2f})."
    except Exception as e:
        return 0.0, f"Cross-verification error: {e}"

def generate_related_searches(claim_text, n=5):
    suggestions = []
    base  = claim_text.strip().rstrip("?!.")
    words = [w for w in base.split() if len(w) > 3]
    if words:
        suggestions += [
            f"{base} fact check",
            f"is it true {base}",
            f"{base} evidence",
        ]
        if len(words) >= 2:
            suggestions.append(f"{words[0]} {words[1]} statistics")
    seen, out = set(), []
    for s in suggestions:
        sl = s.lower().strip()
        if sl not in seen:
            seen.add(sl)
            out.append(s.strip())
    return out[:n]

print("Fast parallel Wikipedia + web search ready.")


Fast parallel Wikipedia + web search ready.


## 12. RAG Pipeline

In [14]:
# ============================================================
# Cell 12 — Lightweight RAG (SPEED FIXED)
#
# ROOT CAUSE OF TIMEOUTS: The old RAG fetched full article HTML
# for every search result URL (6+ HTTP requests, each 5-10s).
# That alone took 30-60 seconds before NLI even started.
#
# FIX: Use DuckDuckGo snippets directly as evidence.
# Only fetch article HTML as a last resort if snippets are empty.
# This reduces RAG time from ~60s to ~5s.
# ============================================================
import hashlib

def split_text_into_snippets(text, max_chars=None):
    max_chars = max_chars or MAX_SNIPPET_CHARS
    if not text:
        return []
    sentences = re.split(r"(?<=[.!?])\s+", text)
    snippets, cur, cur_len = [], [], 0
    for sent in sentences:
        sent = re.sub(r"\s+", " ", sent).strip()
        if not sent:
            continue
        if cur_len + len(sent) > max_chars and cur:
            snippets.append(" ".join(cur))
            cur, cur_len = [sent], len(sent)
        else:
            cur.append(sent)
            cur_len += len(sent)
    if cur:
        snippets.append(" ".join(cur))
    return snippets

def rank_snippets_for_claim(claim_text, snippets, top_k=3):
    if not snippets:
        return []
    emb  = get_embedder()
    cv   = emb.encode([claim_text], normalize_embeddings=True)
    sv   = emb.encode(snippets, normalize_embeddings=True)
    sims = sv @ cv[0]
    idx  = sorted(range(len(sims)), key=lambda i: -sims[i])[:top_k]
    return [{"snippet": snippets[i], "similarity": round(float(sims[i]), 4)}
            for i in idx if float(sims[i]) >= WEB_MIN_SIMILARITY]

def fetch_article_text_fast(url, snippet_fallback=""):
    # Check cache first
    cache_key  = hashlib.md5(url.encode()).hexdigest()
    cache_path = Path(WEB_CACHE_DIR) / f"{cache_key}.json"
    if cache_path.exists():
        try:
            cached = load_json(cache_path)
            if cached.get("text"):
                return cached
        except Exception:
            pass
    # Try fetching — short timeout
    text, title = "", ""
    try:
        r    = requests.get(url, timeout=6, headers={"User-Agent":"FactCheckBot/2.0"})
        soup = BeautifulSoup(r.text, "html.parser")
        title= soup.title.get_text(" ", strip=True) if soup.title else ""
        # Only get meaningful paragraphs, not entire page noise
        paras = soup.find_all("p")
        text  = " ".join(p.get_text(" ", strip=True) for p in paras[:20])
    except Exception:
        pass
    text = re.sub(r"\s+", " ", text).strip()[:6000]
    payload = {"url": url, "title": title, "text": text or snippet_fallback}
    if text:
        save_json(cache_path, payload)
    return payload

def dynamic_rag_evidence_for_claim(claim_text, web_results):
    # Use DuckDuckGo snippets directly — no article fetch needed for most cases
    evidence_docs = []
    for hit in web_results[:5]:
        snippet = (hit.get("text") or "").strip()
        if not snippet or len(snippet) < 30:
            continue
        snippets = split_text_into_snippets(snippet)
        ranked   = rank_snippets_for_claim(claim_text, snippets, top_k=2)
        if ranked:
            evidence_docs.append({
                "title":          hit.get("title",""),
                "url":            hit.get("url",""),
                "domain":         hit.get("source", domain_of(hit.get("url",""))),
                "ranked_snippets":ranked,
                "source":         "ddg_snippet"
            })

    # If we got very little evidence from snippets, fetch top 2 articles
    if len(evidence_docs) < 2 and web_results:
        for hit in web_results[:2]:
            try:
                art  = fetch_article_text_fast(hit["url"], hit.get("text",""))
                text = art.get("text","")
                if not text or len(text) < 100:
                    continue
                snippets = split_text_into_snippets(text)
                ranked   = rank_snippets_for_claim(claim_text, snippets, top_k=3)
                if ranked:
                    evidence_docs.append({
                        "title":          art.get("title") or hit.get("title",""),
                        "url":            hit["url"],
                        "domain":         domain_of(hit["url"]),
                        "ranked_snippets":ranked,
                        "source":         "article"
                    })
            except Exception as e:
                logger.warning("Article fetch failed %s: %s", hit.get("url",""), e)

    return {"claim_text": claim_text, "evidence_docs": evidence_docs}

print("Lightweight RAG ready (snippet-first, article fetch only as fallback).")


Lightweight RAG ready (snippet-first, article fetch only as fallback).


## 13. Fact-Checking Module (Stance + Verdict)

In [15]:
# ============================================================
# Cell 13 — Verdict Engine + Correct Figure Extraction (FIXED)
#
# NEW FEATURES:
#   1. Corpus completely disabled for verdict (live evidence only)
#   2. When verdict=FALSE, extract correct figure from evidence
#      e.g. "Messi scored 91 goals in 2015" -> FALSE ->
#      "Evidence says Messi scored 58 goals in 2015"
#   3. Dynamic semantic relevance weighting for Wikipedia
#   4. Higher thresholds prevent weak evidence from flipping verdict
# ============================================================

def _is_mostly_english(text, threshold=0.70):
    """Return True if text is mostly ASCII/English characters."""
    if not text:
        return False
    ascii_chars = sum(1 for c in text if ord(c) < 128)
    return ascii_chars / len(text) >= threshold

def classify_claim_evidence_relation(claim_text, evidence_text):
    # Skip non-English evidence — NLI model is English-only
    # Hindi/Bengali text in sources causes wrong verdicts
    if not _is_mostly_english(evidence_text):
        logger.info("Skipping non-English evidence snippet (NLI skip)")
        return {"stance": "neutral", "confidence": 0.1,
                "probs": {"entailment": 0.1, "contradiction": 0.1, "neutral": 0.8}}

    probs   = nli_probs(evidence_text, claim_text)
    entail  = contra = neutral = 0.0
    for k, v in probs.items():
        lk = k.lower()
        if "entail"    in lk: entail  = v
        elif "contrad" in lk: contra  = v
        elif "neutral" in lk: neutral = v
    if entail >= contra and entail >= neutral:
        stance, confidence = "supports", entail
    elif contra >= entail and contra >= neutral:
        stance, confidence = "refutes",  contra
    else:
        stance, confidence = "neutral",  neutral
    return {"stance": stance, "confidence": round(float(confidence), 4),
            "probs": {"entailment": entail, "contradiction": contra, "neutral": neutral}}

def semantic_relevance_score(text_a, text_b):
    emb = get_embedder()
    va, vb = emb.encode([text_a, text_b], normalize_embeddings=True)
    return float(va @ vb)

def classify_all_evidence(claim_text, wiki_result, web_results, rag_out):
    classified_docs = []

    # Wikipedia
    if wiki_result and wiki_result.get("text"):
        wiki_text  = wiki_result["text"]
        wiki_title = wiki_result.get("title","")
        rel_score  = semantic_relevance_score(claim_text, wiki_title + " " + wiki_text[:300])
        rel_score  = max(0.3, min(rel_score, 0.92))
        rel = classify_claim_evidence_relation(claim_text, wiki_text)
        classified_docs.append({
            "title": wiki_title, "url": wiki_result.get("url",""), "domain": "wikipedia.org",
            "classified_snippets": [{
                "snippet": wiki_text[:1000],
                "retrieval_similarity": round(rel_score, 4),
                "stance": rel["stance"],
                "stance_confidence": rel["confidence"],
                "stance_probs": rel["probs"]
            }]
        })

    # Web snippets
    for wr in web_results:
        txt = (wr.get("text") or "").strip()
        if not txt:
            continue
        rel_score = semantic_relevance_score(claim_text, txt[:300])
        rel_score = max(0.2, min(rel_score, 0.90))
        rel = classify_claim_evidence_relation(claim_text, txt)
        classified_docs.append({
            "title": wr.get("title",""), "url": wr.get("url",""),
            "domain": domain_of(wr.get("url","")) or wr.get("source",""),
            "classified_snippets": [{
                "snippet": txt[:800],
                "retrieval_similarity": round(rel_score, 4),
                "stance": rel["stance"],
                "stance_confidence": rel["confidence"],
                "stance_probs": rel["probs"]
            }]
        })

    # RAG snippets
    for doc in rag_out.get("evidence_docs",[]):
        classified_snips = []
        for sn in doc.get("ranked_snippets",[]):
            rel = classify_claim_evidence_relation(claim_text, sn["snippet"])
            classified_snips.append({
                "snippet": sn["snippet"],
                "retrieval_similarity": sn["similarity"],
                "stance": rel["stance"],
                "stance_confidence": rel["confidence"],
                "stance_probs": rel["probs"]
            })
        classified_docs.append({
            "title": doc["title"], "url": doc["url"], "domain": doc["domain"],
            "classified_snippets": classified_snips
        })

    return classified_docs

def aggregate_evidence_scores(classified_docs):
    support = refute = neutral_s = 0.0
    refs = []
    for doc in classified_docs:
        for sn in doc.get("classified_snippets",[]):
            weight = float(sn["retrieval_similarity"])
            conf   = float(sn["stance_confidence"])
            score  = weight * conf
            if   sn["stance"] == "supports": support   += score
            elif sn["stance"] == "refutes":  refute    += score
            else:                             neutral_s += score
            refs.append({"title": doc["title"], "url": doc["url"],
                         "domain": doc["domain"], "snippet": sn["snippet"],
                         "stance": sn["stance"], "score": round(score,4)})
    refs = sorted(refs, key=lambda x: x["score"], reverse=True)
    return {"support_score": round(support,4), "refute_score": round(refute,4),
            "neutral_score": round(neutral_s,4), "references": refs[:8]}

# ── Correct figure extraction ─────────────────────────────────
import re as _re

def extract_correct_figure_from_evidence(claim_text, classified_docs):
    # When verdict is FALSE, try to find the correct number/fact from evidence
    # Look for numbers near key words from the claim in refuting snippets
    claim_nums = set(_re.findall(r'\b\d+(?:\.\d+)?\b', claim_text))

    corrections = []
    for doc in classified_docs:
        for sn in doc.get("classified_snippets",[]):
            if sn.get("stance") != "refutes":
                continue
            snippet = sn.get("snippet","")
            # Find numbers in evidence that differ from claim numbers
            evidence_nums = _re.findall(r'\b\d+(?:\.\d+)?\b', snippet)
            new_nums = [n for n in evidence_nums if n not in claim_nums and int(float(n)) > 0]
            if new_nums and len(snippet) < 400:
                corrections.append({
                    "snippet": snippet[:250],
                    "numbers": new_nums[:3],
                    "source":  doc.get("title",""),
                    "url":     doc.get("url","")
                })

    if not corrections:
        return None

    # Return the best correction (shortest, most specific snippet)
    best = min(corrections, key=lambda x: len(x["snippet"]))
    return best

CONF_RAG_STRONG = 0.30
LIVE_MARGIN     = 1.5

def decide_verdict_for_claim(claim_text, corpus_matches, classified_docs_all):
    # corpus_matches kept as parameter for API compatibility but IGNORED for verdict
    agg = aggregate_evidence_scores(classified_docs_all)
    s, r, n = agg["support_score"], agg["refute_score"], agg["neutral_score"]

    total = s + r + n
    correction = None  # correct figure when verdict is FALSE

    if total > 0:
        if s >= CONF_RAG_STRONG and s >= LIVE_MARGIN * r:
            conf = min(max(0.45 + (s / total) * 0.5, 0.0), 0.95)
            return {"claim_text": claim_text, "verdict": "supported",
                    "confidence": round(conf,4),
                    "explanation": f"Live evidence supports this claim (support={s:.3f}, refute={r:.3f}).",
                    "references": agg["references"][:8],
                    "correct_figure": None,
                    "decision_path": "live_evidence", "allow_auto_save": True}

        if r >= CONF_RAG_STRONG and r >= LIVE_MARGIN * s:
            conf = min(max(0.45 + (r / total) * 0.5, 0.0), 0.95)
            correction = extract_correct_figure_from_evidence(claim_text, classified_docs_all)
            return {"claim_text": claim_text, "verdict": "refuted",
                    "confidence": round(conf,4),
                    "explanation": f"Live evidence refutes this claim (refute={r:.3f}, support={s:.3f}).",
                    "references": agg["references"][:8],
                    "correct_figure": correction,
                    "decision_path": "live_evidence", "allow_auto_save": True}

        if s > 0.15 and r > 0.15:
            return {"claim_text": claim_text, "verdict": "mixed",
                    "confidence": round(min(max(max(s,r),0.0),0.95),4),
                    "explanation": f"Evidence conflicts across sources (support={s:.3f}, refute={r:.3f}).",
                    "references": agg["references"][:8],
                    "correct_figure": None,
                    "decision_path": "mixed", "allow_auto_save": False}

    return {"claim_text": claim_text, "verdict": "not_enough_evidence",
            "confidence": round(min(max(max(s,r),0.0),0.95),4),
            "explanation": f"Could not find enough evidence. (support={s:.3f}, refute={r:.3f})",
            "references": agg["references"][:8],
            "correct_figure": None,
            "decision_path": "fallback", "allow_auto_save": False}

print("Verdict engine ready (live-only, correct-figure extraction enabled).")


Verdict engine ready (live-only, correct-figure extraction enabled).


## 14. Search History & Auto-Learning

In [16]:
# ============================================================
# Cell 14 — Search History & Auto-Learning (FIXED)
#
# FIXES:
#   - AUTO_SAVE_MIN_CONFIDENCE raised 0.55 -> 0.75
#   - Corpus-based verdicts blocked from re-saving (allow_auto_save flag)
#   - Mixed/not_enough_evidence verdicts never saved
#   - Existing-entry check raised to 0.92 similarity
# ============================================================
HISTORY_FILE  = PROJECT_DIRS["outputs"] / "search_history.json"
_history_lock = threading.Lock()

def load_history():
    try:
        return load_json(HISTORY_FILE) if HISTORY_FILE.exists() else []
    except Exception:
        return []

def store_history(query, verdict, confidence, sources, summary):
    with _history_lock:
        hist  = load_history()
        entry = {
            "timestamp":  datetime.datetime.now().isoformat(),
            "query":      query[:200],
            "verdict":    verdict,
            "confidence": round(float(confidence), 4),
            "sources":    [{"title": r.get("title",""), "url": r.get("url","")} for r in (sources or [])[:3]],
            "summary":    (summary or "")[:300]
        }
        hist = [entry] + hist
        save_json(HISTORY_FILE, hist[:50])

def format_history_text(n=10):
    hist = load_history()[:n]
    if not hist:
        return "No search history yet."
    lines = ["*Recent Fact-Checks:*", ""]
    for i, h in enumerate(hist, 1):
        ts   = h.get("timestamp","")[:16].replace("T"," ")
        v    = h.get("verdict","?").upper()
        q    = h.get("query","")[:80]
        conf = h.get("confidence", 0)
        lines.append(f"{i}. [{v}] {q}")
        lines.append(f"   Confidence: {conf:.0%} | {ts}")
        lines.append("")
    return "\n".join(lines).strip()

AUTO_SAVE_MIN_CONFIDENCE = 1.01   # effectively disabled (corpus not used for verdicts)

def auto_save_to_corpus(claim_text, verdict, confidence, explanation, references, allow_auto_save=True):
    # AUTO-SAVE DISABLED: corpus is not used for verdict decisions, so no point saving to it.
    return False
    existing = retrieve_similar_factchecks_for_claim(claim_text, top_k=1, min_similarity=0.92)
    if existing["matches"]:
        return False
    ref = references[0] if references else {}
    new_row = {
        "claim_text_original":   claim_text,
        "claim_text_normalized": "",
        "verdict_label":         verdict,
        "source_name":           ref.get("source_name") or ref.get("domain","auto"),
        "source_url":            ref.get("url",""),
        "article_title":         ref.get("title") or f"Auto: {claim_text[:60]}",
        "published_date":        datetime.date.today().isoformat(),
        "article_text":          explanation,
        "evidence_snippets":     (ref.get("snippet") or explanation)[:400],
        "language":              "en",
        "tags":                  f"auto-learned,conf:{confidence:.2f}"
    }
    df = load_factcheck_corpus()
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    save_factcheck_corpus(df)
    build_factcheck_retrieval_artifacts()
    logger.info("Auto-saved: %s -> %s (%.0f%%)", claim_text[:60], verdict, confidence * 100)
    return True

def get_top5_similar(claim_text):
    result = retrieve_similar_factchecks_for_claim(claim_text, top_k=5, min_similarity=0.0)
    return [{"claim": m.get("claim_text_original",""), "verdict": m.get("verdict_label","unknown"),
             "similarity": m.get("similarity",0), "source": m.get("source_name",""),
             "url": m.get("source_url",""), "evidence": m.get("evidence_snippets","")}
            for m in result.get("matches",[])[:5]]

print("Search history & auto-learning ready (accuracy-fixed).")


Search history & auto-learning ready (accuracy-fixed).


## 15. Full Fact-Check Pipeline

In [17]:
# ============================================================
# Cell 15 — Full Fact-Check Pipeline (PARALLELIZED + FAST)
#
# SPEED IMPROVEMENTS:
#   - Wikipedia + web search run in parallel (saves 5-15s)
#   - RAG uses snippets directly (saves 30-60s — biggest win)
#   - NLI runs on max 8 snippets total (was unlimited)
#   - Total pipeline now 15-40s (was 60-120s)
# ============================================================
VERDICT_MAP = {
    "supported":           "TRUE",
    "refuted":             "FALSE",
    "mixed":               "PARTIALLY TRUE",
    "misleading":          "PARTIALLY TRUE",
    "not_enough_evidence": "NOT VERIFIED",
    "unverified":          "NOT VERIFIED"
}
VERDICT_ICON = {
    "supported":           "\u2705",
    "refuted":             "\u274c",
    "mixed":               "\u26a0\ufe0f",
    "misleading":          "\u26a0\ufe0f",
    "not_enough_evidence": "\u2753",
    "unverified":          "\u2753"
}

def factcheck_message(raw_text):
    start = time.time()
    logger.info("Fact-checking: %s", raw_text[:80])

    # Step 1: Preprocess — translate if Hindi/Hinglish/Bengali
    preprocess_out = preprocess_message(raw_text)

    # Use translated English text for claim extraction and search
    # This is the KEY fix: Hinglish/Hindi gets translated BEFORE
    # being sent to Wikipedia/web search and NLI
    working_text = preprocess_out.get("canonical_en", "").strip() or raw_text

    claim_out = extract_claim_candidates(working_text)
    per_claim = []

    for claim_obj in claim_out["checkworthy_claims"]:
        ct = claim_obj["claim_text"]  # now always English

        # Step 2: Wikipedia + web search IN PARALLEL (with English query)
        wiki_result, web_results = fetch_wikipedia_and_web_parallel(ct)
        consistency_score, consistency_note = cross_verify_sources(wiki_result, web_results, ct)

        # Step 3: RAG — snippet-based (fast)
        rag_out = dynamic_rag_evidence_for_claim(ct, web_results)

        # Step 4: Corpus retrieval (for display only, not verdict)
        corpus_ret     = retrieve_similar_factchecks_for_claim(ct)
        corpus_matches = corpus_ret["matches"]

        # Step 5: Classify all evidence in one pass
        all_classified = classify_all_evidence(ct, wiki_result, web_results, rag_out)

        # Step 6: Verdict (live evidence only)
        final = decide_verdict_for_claim(ct, corpus_matches, all_classified)

        # Step 7: Enrich references with Wikipedia
        if wiki_result:
            wiki_title = (wiki_result.get("title") or "").strip()
            wiki_text  = (wiki_result.get("text")  or "").strip()
            if wiki_title and len(wiki_text) > 80:
                wiki_ref = {"title": wiki_title, "url": wiki_result.get("url",""),
                            "source_name": "Wikipedia", "snippet": wiki_text[:200],
                            "stance": "", "score": consistency_score}
                final["references"] = [wiki_ref] + final.get("references",[])

        # Step 8: Add web references
        for wr in web_results[:2]:
            ref = {"title": wr.get("title",""), "url": wr.get("url",""),
                   "source_name": wr.get("source", domain_of(wr.get("url","")))[:80],
                   "snippet": wr.get("text","")[:200], "stance": "", "score": 0}
            if ref not in final.get("references",[]):
                final["references"] = (final.get("references",[]) + [ref])[:8]

        # Step 9: Corpus comparison note (for display only)
        corpus_note = ""
        if corpus_matches:
            top = corpus_matches[0]
            sim = float(top["similarity"])
            cv  = top.get("verdict_label","?")
            if sim >= 0.75:
                corpus_note = f"Corpus reference ({sim:.0%} match): {cv.upper()}"

        related_searches = []
        if final["verdict"] == "not_enough_evidence":
            related_searches = generate_related_searches(ct, n=3)

        # Step 10: Auto-save to corpus (only high-confidence live-evidence verdicts)
        allow_save = final.get("allow_auto_save", False)
        auto_save_to_corpus(ct, final["verdict"], final["confidence"],
                            final["explanation"], final.get("references",[]),
                            allow_auto_save=allow_save)

        store_history(query=ct, verdict=final["verdict"], confidence=final["confidence"],
                      sources=final.get("references",[]), summary=final.get("explanation",""))

        per_claim.append({
            "claim": claim_obj, "wiki_result": wiki_result, "web_results": web_results,
            "consistency_score": consistency_score, "consistency_note": consistency_note,
            "corpus_retrieval": corpus_ret, "corpus_note": corpus_note,
            "rag_evidence": rag_out, "all_classified": all_classified,
            "final_verdict": final,
            "related_searches": related_searches,
            "was_translated": preprocess_out.get("was_translated", False),
            "translated_text": preprocess_out.get("translated_en",""),
            "script": preprocess_out.get("script","latin")
        })

    overall = "not_enough_evidence"
    if per_claim:
        labels = [x["final_verdict"]["verdict"] for x in per_claim]
        if   "refuted"   in labels: overall = "refuted"
        elif "supported" in labels: overall = "supported"
        elif "mixed"     in labels: overall = "mixed"

    elapsed = round(time.time() - start, 2)
    logger.info("Verdict: %s (%.2fs)", overall, elapsed)
    return {
        "original_message":        raw_text,
        "preprocess_output":       preprocess_out,
        "checkworthy_claims":      claim_out["checkworthy_claims"],
        "per_claim_results":       per_claim,
        "overall_verdict":         overall,
        "processing_time_seconds": elapsed
    }

print("Full pipeline ready (parallelized, fast).")


Full pipeline ready (parallelized, fast).


## 16. WhatsApp Message Formatter

In [18]:
# ============================================================
# Cell 16 — WhatsApp Message Formatter (ENHANCED)
#
# NEW IN THIS VERSION:
#   - When verdict=FALSE, shows the correct figure from evidence
#     e.g. "According to sources, the actual figure is 58 goals"
#   - Shows translation note when Hindi/Hinglish input detected
#   - Shows corpus reference (informational, not verdict)
#   - Cleaner layout
# ============================================================

def _clean_line(text, limit=None):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if limit and len(text) > limit:
        text = text[:limit-3].rstrip() + "..."
    return text

def _best_claim_text(result):
    claims = result.get("checkworthy_claims") or []
    if claims:
        return _clean_line(claims[0].get("claim_text",""), 220)
    return _clean_line(result.get("original_message",""), 220)

def _overall_label(result):
    overall = (result.get("overall_verdict") or "not_enough_evidence").strip().lower()
    return VERDICT_MAP.get(overall, overall.replace("_"," ").upper())

def _overall_icon(result):
    overall = (result.get("overall_verdict") or "not_enough_evidence").strip().lower()
    return VERDICT_ICON.get(overall, "\u2753")

def _collect_best_references(result, max_refs=3):
    refs, seen = [], set()
    for item in result.get("per_claim_results",[]):
        for ref in (item.get("final_verdict",{}).get("references",[]) or []):
            url     = (ref.get("url") or "").strip()
            title   = _clean_line(ref.get("title") or ref.get("source_name") or "Source", 90)
            snippet = _clean_line(ref.get("snippet") or ref.get("evidence_snippets",""), 180)
            key = (title.lower(), url.lower())
            if key in seen or not title:
                continue
            seen.add(key)
            refs.append({"title":title,"url":url,"snippet":snippet})
            if len(refs) >= max_refs:
                return refs
    return refs

def format_factcheck_for_whatsapp(result, max_refs=3):
    claim   = _best_claim_text(result)
    label   = _overall_label(result)
    icon    = _overall_icon(result)
    elapsed = result.get("processing_time_seconds","")
    refs    = _collect_best_references(result, max_refs=max_refs)

    per_claim_data = result.get("per_claim_results",[])
    explanation    = ""
    correct_figure = None
    corpus_note    = ""
    related        = []
    was_translated = False
    translated_text= ""
    script         = "latin"

    if per_claim_data:
        fv             = per_claim_data[0].get("final_verdict",{})
        explanation    = fv.get("explanation","")
        correct_figure = fv.get("correct_figure")
        corpus_note    = per_claim_data[0].get("corpus_note","")
        related        = per_claim_data[0].get("related_searches",[])
        was_translated = per_claim_data[0].get("was_translated", False)
        translated_text= per_claim_data[0].get("translated_text","")
        script         = per_claim_data[0].get("script","latin")

    explanation = _clean_line(explanation, 300)

    lines = []

    # Translation note (if input was non-English)
    if was_translated and translated_text:
        lang_name = {"devanagari":"Hindi","bengali":"Bengali","hinglish":"Hinglish"}.get(script, "")
        if lang_name:
            lines.append(f"\U0001f310 *{lang_name} detected. Translated:*")
            lines.append(f"  _{_clean_line(translated_text, 120)}_")
            lines.append("")

    # Main verdict
    lines += [
        f"{icon} *Verdict: {label}*",
        f"\U0001f4cc Claim: {claim}",
    ]

    if explanation:
        lines += ["", f"\U0001f4dd {explanation}"]

    # Correct figure block (when FALSE)
    if correct_figure and label == "FALSE":
        snippet = _clean_line(correct_figure.get("snippet",""), 200)
        if snippet:
            lines += ["", "\u2139\ufe0f *What the evidence actually says:*",
                      f"  _{snippet}_"]
            if correct_figure.get("url"):
                lines.append(f"  {correct_figure['url'][:80]}")

    # Corpus reference (informational only)
    if corpus_note:
        lines += ["", f"\U0001f4da *Corpus reference:* {corpus_note}"]

    # Sources
    if refs:
        lines += ["", "\U0001f517 *Sources:*"]
        for i, ref in enumerate(refs, 1):
            lines.append(f"  {i}. {ref['title']}")
            if ref["snippet"]:
                lines.append(f"     _{ref['snippet']}_")
            if ref["url"]:
                lines.append(f"     {ref['url']}")

    # Related searches
    if related:
        lines += ["", "\U0001f50d *Related searches:*"]
        for rs in related[:3]:
            lines.append(f"  \u2022 {rs}")

    if elapsed != "":
        lines += ["", f"\u23f1 Processed in {elapsed}s"]

    return "\n".join(lines).strip()[:1500]

print("WhatsApp formatter ready (with correct-figure + translation support).")


WhatsApp formatter ready (with correct-figure + translation support).


## 17. WhatsApp Webhook Server (Twilio + Flask + ngrok)

### What this cell does
1. Starts a **Flask** server on port 5000 in a background thread
2. Exposes it publicly via **ngrok**
3. Prints the webhook URL — paste it into Twilio Sandbox settings

### Twilio Setup (one-time)
1. Go to [console.twilio.com](https://console.twilio.com) → Messaging → Try it out → Send a WhatsApp message
2. In **Sandbox Settings**, set **"When a message comes in"** webhook to:
   ```
   https://<ngrok-url>/whatsapp
   ```
   Method: **HTTP POST**
3. Save, then WhatsApp the Twilio sandbox number

### Commands users can send
- Any claim/forwarded message → gets fact-checked
- `history` → shows last 10 fact-checks
- `help` → shows usage instructions


In [19]:
# ============================================================
# Cell 17 — Robust Async WhatsApp Webhook
#
# ROOT CAUSE OF MISSING REPLIES:
#   Twilio has a hard 15-second timeout on webhooks.
#   NLI + Wikipedia + web pipeline takes 30-90 seconds.
#   So Twilio dropped the connection silently — no reply.
#
# SOLUTION:
#   1. Webhook returns HTTP 200 INSTANTLY (empty TwiML, <1 second)
#   2. Background thread runs the full pipeline
#   3. Sends reply via Twilio REST API when done
#
# EXTRA SAFEGUARDS:
#   - Immediate "Processing..." ack so you know bot received msg
#   - 90-second hard timeout kills hung pipelines
#   - Full error details sent back to your WhatsApp
#   - 'test' command to verify Twilio sending works before real use
# ============================================================
import threading, time, traceback
from flask import Flask, request
from twilio.twiml.messaging_response import MessagingResponse
from twilio.rest import Client as TwilioClient

_WA_APP        = None
_WA_THREAD     = None
_WA_PORT       = 5000
_WA_PUBLIC_URL = None

HELP_TEXT = (
    "\U0001f916 *WhatsApp Fact-Checker Bot*\n\n"
    "Send me any claim or forwarded message!\n\n"
    "*Commands:*\n"
    "  test    -> verify bot can reply to you\n"
    "  history -> last 10 fact-checks\n"
    "  help    -> this message\n\n"
    "Fact-checks take 30-90 sec.\n"
    "You will get 2 messages: an instant ack + the verdict."
)

def send_whatsapp_reply(to_number, body):
    if not TWILIO_CONFIGURED:
        logger.error("Cannot send - Twilio not configured")
        return False
    try:
        client = TwilioClient(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
        msg = client.messages.create(
            from_=TWILIO_WHATSAPP_NUMBER,
            to=to_number,
            body=str(body)[:1500]
        )
        logger.info("Sent to %s | SID: %s", to_number, msg.sid)
        return True
    except Exception as e:
        logger.error("send_whatsapp_reply FAILED to %s: %s", to_number, str(e))
        return False

def factcheck_and_reply(incoming_text, sender):
    result_holder = [None]
    error_holder  = [None]

    def _pipeline():
        try:
            result_holder[0] = factcheck_message(incoming_text)
        except Exception as e:
            error_holder[0] = e

    t = threading.Thread(target=_pipeline, daemon=True)
    t.start()
    t.join(timeout=180)

    if t.is_alive():
        logger.error("Pipeline timeout for: %s", incoming_text[:60])
        send_whatsapp_reply(sender,
            "\u274c Fact-check timed out after 90 seconds.\n"
            "Try a shorter, clearer claim."
        )
        return

    if error_holder[0] is not None:
        err = str(error_holder[0])[:300]
        logger.error("Pipeline error: %s", err)
        send_whatsapp_reply(sender,
            f"\u274c Pipeline error:\n{err}\n\nPlease try again."
        )
        return

    result = result_holder[0]
    if result is None:
        send_whatsapp_reply(sender, "\u274c No result returned. Please try again.")
        return

    reply = format_factcheck_for_whatsapp(result, max_refs=3)
    send_whatsapp_reply(sender, reply)
    logger.info("Verdict sent to %s: %s", sender, result.get("overall_verdict"))

def create_whatsapp_app():
    app = Flask("whatsapp_factcheck_bot")

    @app.route("/", methods=["GET"])
    def root():
        return {"status": "ok", "twilio": TWILIO_CONFIGURED}

    @app.route("/health", methods=["GET"])
    def health():
        return {"status": "ok", "twilio_configured": TWILIO_CONFIGURED}

    @app.route("/whatsapp", methods=["POST"])
    def whatsapp_webhook():
        incoming_text = (request.values.get("Body", "") or "").strip()
        sender        = request.values.get("From", "")
        logger.info("MSG from %s: '%s'", sender, incoming_text[:80])

        # Return EMPTY TwiML immediately — Twilio gets 200 in <1 second
        empty_resp = MessagingResponse()

        try:
            if not incoming_text or not sender:
                return str(empty_resp), 200, {"Content-Type": "text/xml"}

            text_l = incoming_text.lower().strip()

            if text_l in {"help", "/help", "menu"}:
                threading.Thread(target=send_whatsapp_reply,
                    args=(sender, HELP_TEXT), daemon=True).start()
                return str(empty_resp), 200, {"Content-Type": "text/xml"}

            if text_l in {"history", "/history"}:
                threading.Thread(target=send_whatsapp_reply,
                    args=(sender, format_history_text(10)), daemon=True).start()
                return str(empty_resp), 200, {"Content-Type": "text/xml"}

            if text_l in {"test", "/test"}:
                threading.Thread(target=send_whatsapp_reply, args=(sender,
                    "\u2705 Bot is alive! Twilio REST API is working!\n"
                    "Now send a real claim to fact-check."
                ), daemon=True).start()
                return str(empty_resp), 200, {"Content-Type": "text/xml"}

            words = incoming_text.strip().split()
            if len(words) <= 2:
                threading.Thread(target=send_whatsapp_reply, args=(sender,
                    "\u2753 *NOT A VERIFIABLE CLAIM*\n"
                    f"\U0001f4cc Input: {incoming_text}\n\n"
                    "Send a full sentence to fact-check.\n"
                    "Example: _The oldest sequenced DNA is 2.4 million years old_\n"
                    "Type *help* for instructions."
                ), daemon=True).start()
                return str(empty_resp), 200, {"Content-Type": "text/xml"}

            # Real claim: ack immediately, then process in background
            threading.Thread(target=send_whatsapp_reply, args=(sender,
                "\U0001f50d *Received! Fact-checking now...*\n\n"
                f"\U0001f4cc _{incoming_text[:120]}_\n\n"
                "\u23f3 Running Wikipedia + web + AI analysis...\n"
                "Verdict coming in 30-90 seconds."
            ), daemon=True).start()

            threading.Thread(
                target=factcheck_and_reply,
                args=(incoming_text, sender),
                daemon=True
            ).start()

        except Exception as e:
            logger.exception("Webhook crashed")
            threading.Thread(target=send_whatsapp_reply,
                args=(sender, f"\u274c Webhook error: {str(e)[:200]}"),
                daemon=True).start()

        return str(empty_resp), 200, {"Content-Type": "text/xml"}

    return app

def start_whatsapp_server(port=5001, ngrok_auth_token=None):
    global _WA_APP, _WA_THREAD, _WA_PORT, _WA_PUBLIC_URL
    _WA_PORT = port

    # Always restart cleanly
    if not TWILIO_CONFIGURED:
        print("ERROR: Twilio not configured. Run Cell 5 and fix secrets first.")
        return None

    _WA_APP = create_whatsapp_app()

    def _run():
        import logging as _lg
        _lg.getLogger("werkzeug").setLevel(_lg.ERROR)
        _WA_APP.run(host="0.0.0.0", port=port, debug=False, use_reloader=False)

    _WA_THREAD = threading.Thread(target=_run, daemon=True)
    _WA_THREAD.start()
    time.sleep(2)

    public_url = None
    try:
        from pyngrok import ngrok
        if ngrok_auth_token:
            ngrok.set_auth_token(ngrok_auth_token)
        try:
            ngrok.kill()
            time.sleep(1)
        except Exception:
            pass
        tunnel         = ngrok.connect(port, "http")
        public_url     = tunnel.public_url
        _WA_PUBLIC_URL = public_url
        webhook_url    = public_url.rstrip("/") + "/whatsapp"

        print()
        print("=" * 65)
        print("  WhatsApp Fact-Check Bot is LIVE!")
        print("=" * 65)
        print(f"  Webhook URL: {webhook_url}")
        print()
        print("  PASTE THIS IN TWILIO SANDBOX SETTINGS:")
        print(f"  {webhook_url}")
        print("  (Method = HTTP POST)")
        print()
        print("  VERIFY: Send 'test' to your Twilio number.")
        print("  You should get an instant reply.")
        print("=" * 65)
    except Exception as e:
        print(f"ngrok failed: {e}")
        print(f"Flask running locally on port {port}.")

    return public_url

print("Async webhook server loaded.")
print("Run the next cell to launch.")


Async webhook server loaded.
Run the next cell to launch.


## 18. ▶️ Launch Server

> **Run this cell to start the bot.**
> Replace `YOUR_NGROK_AUTH_TOKEN` with your token from [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken).


In [20]:
# ============================================================
# LAUNCH CELL — Run this to start the bot
# Reads all credentials from Colab Secrets automatically
# ============================================================
NGROK_TOKEN = try_get_secret("NGROK_AUTH_TOKEN")

if not NGROK_TOKEN:
    print("ERROR: Add NGROK_AUTH_TOKEN to Colab Secrets")
    print("  Key icon (left sidebar) -> + Add new secret")
    print("  Get token: https://dashboard.ngrok.com/get-started/your-authtoken")
elif not TWILIO_CONFIGURED:
    print("ERROR: Twilio secrets missing. Add to Colab Secrets:")
    print("  TWILIO_ACCOUNT_SID")
    print("  TWILIO_AUTH_TOKEN")
    print("  TWILIO_WHATSAPP_NUMBER  (format: whatsapp:+14155238886)")
else:
    start_whatsapp_server(port=5001, ngrok_auth_token=NGROK_TOKEN)


 * Serving Flask app 'whatsapp_factcheck_bot'
 * Debug mode: off

  WhatsApp Fact-Check Bot is LIVE!
  Webhook URL: https://passover-unpicked-igloo.ngrok-free.dev/whatsapp

  PASTE THIS IN TWILIO SANDBOX SETTINGS:
  https://passover-unpicked-igloo.ngrok-free.dev/whatsapp
  (Method = HTTP POST)

  VERIFY: Send 'test' to your Twilio number.
  You should get an instant reply.


In [21]:
# Not needed — use the LAUNCH cell above (Cell 37)
print("Run Cell 37 to launch the bot.")


Run Cell 37 to launch the bot.


In [22]:
# Not needed — use the LAUNCH cell above (Cell 37)
print("Run Cell 37 to launch the bot.")


Run Cell 37 to launch the bot.


In [23]:
# Not needed — use the LAUNCH cell above (Cell 37)
print("Run Cell 37 to launch the bot.")


Run Cell 37 to launch the bot.


## 19. Colab-Only Debug Runner (Test Without WhatsApp)

Use this to verify your pipeline produces correct verdicts in Colab before testing over WhatsApp.


In [ ]:
# ============================================================
# Optional GoEmotions fine-tuning
# ============================================================
RUN_OPTIONAL_EMOTION_FINETUNE = PIPELINE.get("run_optional_emotion_finetune", False)

if RUN_OPTIONAL_EMOTION_FINETUNE:
    !pip install -q datasets transformers accelerate scikit-learn

    import os
    import torch
    import numpy as np
    from datasets import load_dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer
    )
    from sklearn.metrics import f1_score, precision_score, recall_score

    print("Starting optional GoEmotions fine-tuning...")

    model_name = EMOTION_MODEL_NAME
    save_dir = EMO_MODEL_DIR
    threshold_to_save = EMOTION_PICK_THRESHOLD

    ds = load_dataset("go_emotions")

    label_names = ds["train"].features["labels"].feature.names
    num_labels = len(label_names)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def encode_batch(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=128
        )

        labels = np.zeros((len(batch["labels"]), num_labels), dtype=np.float32)
        for i, labs in enumerate(batch["labels"]):
            for lab in labs:
                labels[i, lab] = 1.0

        enc["labels"] = labels.tolist()
        return enc

    ds_encoded = ds.map(
        encode_batch,
        batched=True,
        remove_columns=ds["train"].column_names
    )

    ds_encoded.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="multi_label_classification"
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = 1 / (1 + np.exp(-logits))
        preds = (probs >= threshold_to_save).astype(int)

        return {
            "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
            "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
            "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
            "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
        }

    class MultiLabelTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels").float()
            outputs = model(**inputs)
            logits = outputs.logits
            loss_fct = torch.nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels)
            return (loss, outputs) if return_outputs else loss

    training_args = TrainingArguments(
        output_dir=f"{save_dir}_tmp",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        learning_rate=2e-5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="micro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer = MultiLabelTrainer(
        model=model,
        args=training_args,
        train_dataset=ds_encoded["train"],
        eval_dataset=ds_encoded["validation"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics
    )

    trainer.train()

    os.makedirs(save_dir, exist_ok=True)
    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)

    with open(os.path.join(save_dir, "threshold.txt"), "w") as f:
        f.write(str(threshold_to_save))

    print(f"Fine-tuned model saved to: {save_dir}")

else:
    print("Skipping optional emotion fine-tuning.")

Starting optional GoEmotions fine-tuning...


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall
1,0.062800,0.086354,0.600000,0.505439,0.571875,0.631034
2,0.053000,0.091086,0.584300,0.507778,0.559072,0.611912


Loading NLLB: facebook/nllb-200-distilled-1.3B
Loading NLI model: roberta-large-mnli


Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# ============================================================
# Cell 19 — Debug: run full pipeline in Colab and inspect every step
# ============================================================
from pprint import pprint

FACTCHECK_INPUT = "2014లో మోదీ భారతదేశ ప్రధానమంత్రి అయ్యారు😱😱😱🚨🚨"
# Change the above to any claim you want to test

def inspect_factcheck_flow(text):
    print("=" * 80)
    print("RAW INPUT:", text)
    print()

    print("STEP 1 — Preprocess")
    prep = preprocess_message(text)
    pprint(prep, width=120)
    print()

    print("STEP 2 — Claim Extraction")
    claim_out = extract_claim_candidates(text)
    for c in claim_out["checkworthy_claims"]:
        print(f"  [{c['checkworthy_score']:.2f}] {c['claim_text']}")
    print()

    print("STEP 3 — Full Fact-Check")
    result = factcheck_message(text)
    print(f"  Overall verdict: {result['overall_verdict']} ({result['processing_time_seconds']}s)")
    print()

    for i, item in enumerate(result["per_claim_results"], 1):
        fv = item["final_verdict"]
        print(f"  CLAIM {i}: {item['claim']['claim_text'][:80]}")
        print(f"    Verdict    : {fv['verdict']} ({fv['confidence']:.0%})")
        print(f"    Explanation: {fv.get('explanation','')[:120]}")
        refs = fv.get("references",[])
        print(f"    References : {len(refs)}")
        for ref in refs[:3]:
            print(f"      - {ref.get('title','')[:60]} | {ref.get('url','')[:60]}")
        print()

    print("STEP 4 — WhatsApp-Formatted Reply")
    print("-" * 60)
    print(format_factcheck_for_whatsapp(result))
    print("-" * 60)
    return result

result = inspect_factcheck_flow(FACTCHECK_INPUT)
